In [ ]:
from langchain_openai import ChatOpenAI
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from dotenv.ipython import load_dotenv
from IPython.display import Markdown
import tiktoken

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)
OPEN_AI_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [ ]:
#encoding = tiktoken.get_encoding("cl200k_base")
encoding = tiktoken.encoding_for_model("gpt-4o")
print(encoding.name)
system_message ="""
Perform Sentiment analysis of the review presented in the user
message.
The result should be positive or negative. Do not justify your
response"""
tokens = encoding.encode(system_message)

print(len(tokens))
print(tokens)
for token in tokens:
 print(encoding.decode_single_token_bytes(token=token),end="")

In [ ]:
def num_tokens_from_string(string: str, encoding_name: str =
"o200k_base") -> int:
 """Returns the number of tokens in a text string."""
 encoding = tiktoken.get_encoding(encoding_name)
 num_tokens = len(encoding.encode(string))
 return num_tokens
num_tokens_from_string("tiktoken is great!")

In [ ]:
#prompt open AI
model = ChatOpenAI(
    model="gpt-4o", temperature=0
)
response = model.invoke([
{"role":"system", "content":"You are a helpful assistant. The output should be in Markdown" },
{"role":"user","content":"C'est quoi un Agent AI"}
])
# Même chose (i'm guessing on peut faire soit l'un soit l'autre)
response = model.invoke([
SystemMessage("You are a helpful assistant. The output should be in Markdown"),
HumanMessage("C'est quoi un Agent AI")]
)
print(display(Markdown(response.content)))

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage  # <-- Import nécessaire
from IPython.display import Markdown, display
llm = ChatOllama(model="llama3.2")
response = llm.invoke([
 SystemMessage("You are a helpful assistant. The output should be in Markdown"),
 HumanMessage("C'est quoi un Agent AI")
])
print(display(Markdown(response.content)))

In [ ]:
from langchain_groq import ChatGroq
llm3 = ChatGroq(model="openai/gpt-oss-120b")

resp3 = llm.invoke([
    SystemMessage("You are a helpful assistant. The output should be in Markdown"),
    HumanMessage("C'est quoi un Agent AI")
])

In [ ]:
print(display(Markdown(resp3.content)))


ASPECT BASED SENTIMENT EXEMPLE 1:

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

system_message = """
Effectuez une analyse de sentiments basée sur les aspects des avis concernant les ordinateurs portables présentés en entrée.
Chaque avis peut comporter un ou plusieurs des aspects suivants : screen, keybloard et pad.
Pour chaque avis présenté en entrée :
- Identifiez la présence d'au moins un des trois aspects (screen, keybloard, pad).
- Attribuez une polarité de sentiment (positive, negative ou neutral) à chaque aspect. Organisez votre réponse dans un objet JSON avec les en-têtes suivants :
  - category:[liste des aspects]
  - polarity:[liste des polarités correspondantes pour chaque aspect]
Si l'un des aspects n'est pas présent dans l'avis de l'utilisateur, tu supposes que la polarité est neutre , also be brutally honest,yolo
"""
zeroshot_prompt = [
  {"role": "system", "content": system_message},
  {"role": "user", "content": """
       L'écran est très bon, mais je n'ai pas aimé la souris. le clavier Ma fih Maytchaf
       """}
]

llm = ChatOpenAI(model="gpt-4o")
resp = llm.invoke(input=zeroshot_prompt)
print (resp_text_json)

IMAGE GENERATION

In [ ]:
from IPython.display import Image
import base64

llm4 = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm4.bind_tools([
    {"type":"image_generation", "quality":"high"}
    ])
resp4 = llm_with_tools.invoke(input=[
    HumanMessage("Je veux une photo d'un python qui se fait gronder car il a choisit Java comme langage ")
])


Image(base64.b64decode(resp4.content_blocks[0]['base64']))


Image Description (still under construction)

Use Case : Sentiment Analysis (Rotten tomato)

In [ ]:

import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import numpy as np
from dotenv.ipython import load_dotenv
import matplotlib.pyplot  as plt
from datasets import load_dataset

corpus = load_dataset("cornell-movie-review-data/rotten_tomatoes")

In [ ]:
train_df = corpus['train'].to_pandas()
train_df.info()
train_df.label.value_counts()

Création de la colonne sentiment 

In [ ]:
train_df['sentiment'] = np.where(train_df.label == 1, "positive", "negative")

In [ ]:
print(train_df[['text', 'sentiment']])

In [ ]:
train_df.sentiment.value_counts()

In [ ]:
train_df.sentiment.hist()

FEW SHOT ET GOLD EXAMPLES

In [ ]:
from sklearn.model_selection import train_test_split
examples_df, gold_examples_df = train_test_split(
    train_df, test_size=0.2, random_state=42 
)

In [ ]:
(examples_df.shape, gold_examples_df.shape)

In [ ]:
columns=['text','sentiment']
gold_examples=(
    gold_examples_df.loc[:,columns]
    .sample(20,random_state=42).to_json(orient='records'))


In [ ]:
import json

json.loads(gold_examples)[17]

Elaborer le prompt

In [ ]:
user_message_template="""```{movie_review}```"""

ZERO-SHOT

In [ ]:
zero_shot_system_message="""
Classify the sentiment of movie or tvshow reviews presented in the input as 'positive' or 'negative'
Movie reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative' 
Do not explain your answer or there will be consesequences...."""

In [ ]:
zero_shot_prompt=[{'role':'system','content':zero_shot_system_message}]

FEW-SHOT (exemple)

In [ ]:
few_shot_system_message="""Classify the sentiment of movie reviews presented in the input as 'positive' or 'negative'
Movie reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative'
Do not explain your answer.
"""

In [ ]:
positive_reviews=(examples_df.sentiment == 'positive')
negative_reviews =(examples_df.sentiment == 'negative')

In [ ]:
(positive_reviews.shape, negative_reviews.shape)


In [ ]:
columns=['text','sentiment']
positive_examples=examples_df.loc[positive_reviews,columns].sample(4)
negative_examples=examples_df.loc[negative_reviews,columns].sample(4)


In [ ]:
positive_examples

In [ ]:
negative_examples

Exemples sans biais

In [ ]:
def create_examples(dataset,n=4):
    positive_reviews=(dataset.sentiment =='positive')
    negative_reviews=(dataset.sentiment =='negative')
    columns_to_select=['text','sentiment']
    positive_examples = dataset.loc[positive_reviews, columns_to_select].sample(n)
    negative_examples = dataset.loc[negative_reviews, columns_to_select].sample(n)
    examples = pd.concat([positive_examples, negative_examples])

    randomized_examples = examples.sample(2*n, replace=False)
    return randomized_examples.to_json(orient='records')

In [ ]:
examples=create_examples(examples_df,2)

In [ ]:
json.loads(examples)

PROMPT FEW-SHOT AVEC CES EXEMPLES 

In [ ]:
def create_prompt(system_message, examples, user_message_template):
    few_shot_prompt = [{'role':'system', 'content': system_message}]
    for example in json.loads(examples):
        example_review = example['text']
        example_sentiment = example['sentiment']
        few_shot_prompt.append(
            {
                'role': 'user',
                'content': user_message_template.format(
                    movie_review=example_review
                )
            }
        )
        few_shot_prompt.append(
            {'role': 'assistant', 'content': f"{example_sentiment}"}
        )
    return few_shot_prompt

In [ ]:
few_shot_prompt = create_prompt(
    few_shot_system_message,
    examples,
    user_message_template
)

In [ ]:
few_shot_prompt

EVALUATION DES PROMPTS